In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision 
from torchvision.datasets import CIFAR10

In [5]:
#datasets and dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
#image=>scale(0,1)=>normalize=>(-1,1)
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
trainset=CIFAR10(root="./data",train=True ,download=True,transform=transform)
testset=CIFAR10(root="./data",train=False ,download=True,transform=transform)

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [01:35<00:00, 1.79MB/s]


In [6]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [7]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)


# build the CNN

In [18]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers=nn.Sequential(
             nn.Conv2d(3,32,kernel_size=3,padding=1),
             nn.ReLU(),
             nn.MaxPool2d(2,2),#kernel size=2,stride size=2
             nn.Conv2d(32,64,kernel_size=3,padding=1),
             nn.ReLU(),
             nn.MaxPool2d(2,2),#kernel size=2,stride size=2
             nn.Conv2d(64,128,kernel_size=3,padding=1),
             nn.ReLU(),
             nn.MaxPool2d(2,2)#kernel size=2,stride size=2
            
        )
        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
                            
        
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)
        return x

In [19]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

# training the CNN

In [21]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    for images ,labels in trainloader:
        optimizer.zero_grad()
        output=model.forward(images)#forward propogation
        loss=criterion(output,labels)# loss calculation
        loss.backward()#bp
        optimizer.step()#update parameters
        epoch_training_loss+=loss.item()
    print(f"{epoch+1}/{epoch} the loss is {epoch_training_loss/len(trainloader)}")

1/0 the loss is 0.7419995103040924
2/1 the loss is 0.612314107053725
3/2 the loss is 0.5038046199266258
4/3 the loss is 0.4086658377248003
5/4 the loss is 0.3188373271823692
6/5 the loss is 0.24704034169159278
7/6 the loss is 0.18821038511079138
8/7 the loss is 0.14299076135791933
9/8 the loss is 0.1241864954166667
10/9 the loss is 0.10255759603479672


# evalution

In [22]:
total_prediction=0
correct_prediction=0
model.eval()
with torch.no_grad():
    for images,labels in testloader:
        output=model.forward(images)
        _,predicted=torch.max(output,1)
        correct_prediction+=(predicted==labels).sum().item()
        total_prediction+=labels.size(0)
print("accuracy=",{correct_prediction/total_prediction})


accuracy= {0.7505}
